## Understanding Buffers
* In essence, PyTorch buffers are tensor attributes associated with a PyTorch module or model similar to parameters, but unlike parameters, buffers are not updated during training.

* Buffers in PyTorch are particularly useful when dealing with GPU computations, as they need to be transferred between devices (like from CPU to GPU) alongside the model's parameters. Unlike parameters, buffers do not require gradient computation, but they still need to be on the correct device to ensure that all computations are performed correctly.

* In chapter 3, we use PyTorch buffers via `self.register_buffer`, which is only briefly explained in the book. Since the concept and purpose are not immediately clear, this code notebook offers a longer explanation with a hands-on example.

## An Example without Buffers

* Suppose we've the following code , which is based on the code from chapter 3. This version has been modified to exulde buffers. It imeplements the casual self-attention mechanism used in LLMs


In [10]:
import torch
import torch.nn as nn


class CasualAttentionWithoutBuffers(nn.Module):

  def __init__(self, d_in, d_out, context_length,
               dropout, qkv_bias=False):

    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    attn_weights = self.dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec


** We can Initialize andrun the module as follows on some example data **



In [11]:
torch.manual_seed(123)

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],   # Your           (x^1)
     [0.55, 0.87, 0.66],   #  journey      (x^2)
     [0.57, 0.85, 0.64],   # starts        (x^3)
     [0.22, 0.58, 0.33],   # with          (x^4)
     [0.77, 0.80, 0.55],    # one           (x^5)
     [0.05, 0.80, 0.55]]    # step          (x^6)
)

batch  = torch.stack((inputs, inputs), dim=0)
context_length = batch.shape[1]
d_in = inputs.shape[1]
d_out = 2


ca_without_buffer = CasualAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)

with torch.no_grad():
  context_vecs = ca_without_buffer(batch)


print(context_vecs)

tensor([[[0.3326, 0.5659],
         [0.3446, 0.5651],
         [0.3434, 0.5607],
         [0.3110, 0.4962],
         [0.3029, 0.5017],
         [0.3127, 0.4900]],

        [[0.3326, 0.5659],
         [0.3446, 0.5651],
         [0.3434, 0.5607],
         [0.3110, 0.4962],
         [0.3029, 0.5017],
         [0.3127, 0.4900]]])


So far, everything has worked fine so far.

However, when training LLMs, we typically use GPUs to accelerate the process. Therefore, let's transfer the `CausalAttentionWithoutBuffers` module onto a GPU device.

Please note that this operation requires the code to be run in an environment equipped with GPUs.

In [13]:
has_cuda = torch.cuda.is_available()
has_mps = torch.backends.mps.is_available()

print("Machine has GPU:", has_cuda  or has_mps)

if has_mps:
  device = torch.device("msp")    # APple silicon GPU (Metal)
elif has_cuda:
  device = torch.device("cuda")     # NVIDIA GPU
else:
  device = torch.device("cpu")

print(f"Using device: {device}")

batch = batch.to(device)
ca_without_buffer = ca_without_buffer.to(device)


Machine has GPU: True
Using device: cuda


In [14]:
with torch.no_grad():
  context_vecs = ca_without_buffer(batch)
print(context_vecs)

RuntimeError: expected self and mask to be on the same device, but got mask on cpu and self on cuda:0

Running the code resulted in an error. What happened? It seems like we attempted a matrix multiplication between a tensor on a GPU and a tensor on a CPU. But we moved the module to the GPU!?

Let's double-check the device locations of some of the tensors.....


In [15]:
print("w_query.device", ca_without_buffer.W_query.weight.device)

print("mask.device", ca_without_buffer.mask.device)

w_query.device cuda:0
mask.device cpu


In [16]:
type(ca_without_buffer.mask)

torch.Tensor

As we can see, the `mask` was not moved onto the GPU. That's because it's not a PyTorch parameter like the weights (e.g., `W_query.weight`).

This means we have to manually move it to the GPU via `.to("cuda")`:

In [17]:
ca_without_buffer.mask = ca_without_buffer.mask.to(device)
print("mask.device:", ca_without_buffer.mask.device)

mask.device: cuda:0


Trying the code again

In [18]:
with torch.no_grad():
  context_vecs = ca_without_buffer(batch)

print(context_vecs)

tensor([[[0.3326, 0.5659],
         [0.3446, 0.5651],
         [0.3434, 0.5607],
         [0.3110, 0.4962],
         [0.3029, 0.5017],
         [0.3127, 0.4900]],

        [[0.3326, 0.5659],
         [0.3446, 0.5651],
         [0.3434, 0.5607],
         [0.3110, 0.4962],
         [0.3029, 0.5017],
         [0.3127, 0.4900]]], device='cuda:0')


This time, it worked!

However, remembering to move individual tensors to the GPU can be tedious. As we will see in the next section, it's easier to use `register_buffer` to register the `mask` as a buffer.

## An example with buffers

Let's now modilfy the casual attention class to register the `mask` as a buffer:


In [26]:
import torch
import torch.nn as nn

class CausalAttentionWithBuffer(nn.Module):

  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)

    # Old::
    # self.mask = torch.triu(torch.ones(context_length. context_length), diagonal=1)

    # New
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2)
    attn_scores.masked_fill(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf

    )
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    attn_weights = self.dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec





Now, conveniently, if we move the module to the GPU, the mask will be located on the GPU as well:


In [27]:
ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
ca_with_buffer.to(device)

print("W_query.dvice", ca_with_buffer.W_query.weight.device)
print("mask.device", ca_with_buffer.mask.device)

W_query.dvice cuda:0
mask.device cuda:0


In [28]:
with torch.no_grad():
  context_vecs = ca_with_buffer(batch)

print(context_vecs)

tensor([[[-0.4417,  0.6770],
         [-0.4436,  0.6683],
         [-0.4436,  0.6683],
         [-0.4422,  0.6690],
         [-0.4441,  0.6673],
         [-0.4420,  0.6695]],

        [[-0.4417,  0.6770],
         [-0.4436,  0.6683],
         [-0.4436,  0.6683],
         [-0.4422,  0.6690],
         [-0.4441,  0.6673],
         [-0.4420,  0.6695]]], device='cuda:0')




As we can see above, registering a tensor as a buffer can make our lives a lot easier: We don't have to remember to move tensors to a target device like a GPU manually.


## **Buffers and state_dict**
* Another advantage of PyTorch buffers, over regular tensors, is that they get included in a model's `state_dict`

* For example, consider the `state_dict` of the causal attention object without buffers


In [29]:
ca_without_buffer.state_dict()

OrderedDict([('W_query.weight',
              tensor([[-0.2354,  0.0191, -0.2867],
                      [ 0.2177, -0.4919,  0.4232]], device='cuda:0')),
             ('W_value.weight',
              tensor([[-0.1362,  0.1853,  0.4083],
                      [ 0.1076,  0.1579,  0.5573]], device='cuda:0')),
             ('W_key.weight',
              tensor([[-0.4900, -0.3503, -0.2120],
                      [-0.1135, -0.4404,  0.3780]], device='cuda:0'))])

* The mask is not included in the `state_dict`
 above

 * However, the mask `is` included in the `state_dict`  below, thanks to registering it as a buffer

In [30]:
ca_with_buffer.state_dict()

OrderedDict([('mask',
              tensor([[0., 1., 1., 1., 1., 1.],
                      [0., 0., 1., 1., 1., 1.],
                      [0., 0., 0., 1., 1., 1.],
                      [0., 0., 0., 0., 1., 1.],
                      [0., 0., 0., 0., 0., 1.],
                      [0., 0., 0., 0., 0., 0.]], device='cuda:0')),
             ('W_query.weight',
              tensor([[ 0.0911,  0.4770, -0.5456],
                      [-0.3887, -0.2299,  0.0232]], device='cuda:0')),
             ('W_value.weight',
              tensor([[-0.1347, -0.0634, -0.5629],
                      [ 0.2704,  0.5068,  0.3529]], device='cuda:0')),
             ('W_key.weight',
              tensor([[-0.4088, -0.4654,  0.2397],
                      [ 0.0130,  0.2367, -0.5642]], device='cuda:0'))])

* A state_dict is useful when saving and loading trained `PyTorch` models, for example
* In this particular case, saving and loading the `mask` is maybe not super useful, because it remains unchanged during training; so, for demonstration purposes, let's assume it was modified where all 1's were changed to 2's:

In [31]:
ca_with_buffer.mask[ca_with_buffer.mask == 1.] = 2.
ca_with_buffer.mask

tensor([[0., 2., 2., 2., 2., 2.],
        [0., 0., 2., 2., 2., 2.],
        [0., 0., 0., 2., 2., 2.],
        [0., 0., 0., 0., 2., 2.],
        [0., 0., 0., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0.]], device='cuda:0')

*

    Then, if we save and load the model, we can see that the mask is restored with the modified value




In [32]:
torch.save(ca_with_buffer.state_dict(), "model.pth")

new_ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
new_ca_with_buffer.load_state_dict(torch.load("model.pth"))

new_ca_with_buffer.mask

tensor([[0., 2., 2., 2., 2., 2.],
        [0., 0., 2., 2., 2., 2.],
        [0., 0., 0., 2., 2., 2.],
        [0., 0., 0., 0., 2., 2.],
        [0., 0., 0., 0., 0., 2.],
        [0., 0., 0., 0., 0., 0.]])

* This is not true if we not use buffers


In [33]:
ca_without_buffer.mask[ca_without_buffer.mask == 1.] == 2.

torch.save(ca_without_buffer.state_dict(),"model.pth")

new_ca_without_buffer = CasualAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)
new_ca_without_buffer.load_state_dict(torch.load("model.pth"))

new_ca_without_buffer.mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])